# A1.7 · Identity spoofing and impersonation

**Function A — AI Architecture, Risks and Mitigations → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.6 · Privilege compromise](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Four agents share one service account. The audit log answers "what happened" perfectly and cannot answer "which one" at all — and neither can the downstream service that was deciding whether to trust the caller.

## 2 · The framework

```
   agent A --+
   agent B --+---> one service account ---> downstream service
   agent C --+          "svc-automation"          |
   agent D --+                                    v
                                        "who called me?"  -> unanswerable

   the audit log is complete and useless: every row has the same subject
```

**OWASP T9 — Identity Spoofing & Impersonation.**

A1.6 was about an agent holding too much authority. This one is about the
**identity** component being unable to say *which agent* is calling at all.

When several agents share one credential — the same service account, the same
API key baked into the same image — they are, to every downstream system, the
same principal. There is no spoofing step required. Impersonation is the
default state, because there was never a distinction to defeat.

Three consequences follow, and the third is the one that hurts during an
incident:

**Authorization cannot differ.** Every agent gets the union of what any of them
needs, which is A1.6 again, arriving through a different door.

**Attribution is impossible.** "Which agent called this?" has no answer. Not a
hard answer — no answer, because the information was never present.

**Revocation is all-or-nothing.** You have one misbehaving agent and one
credential shared by forty. Rotating it stops the incident and stops the other
thirty-nine, so the decision becomes a business call in the middle of a
response, at whatever hour it is.

In a multi-agent topology this compounds. A peer's message is trusted because it
came from a peer — but if identity cannot distinguish peers, "it came from a
peer" is a claim anyone inside the perimeter can make.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Three agents, one credential, one incident.

In [ ]:
SHARED_KEY = "svc-agent-7f3a1c"

AGENTS = {"triage-agent":  {"key": SHARED_KEY},
          "patch-agent":   {"key": SHARED_KEY},
          "deploy-agent":  {"key": SHARED_KEY}}

CALLS = []

def downstream(api_key, action, resource):
    """A downstream service sees only the credential presented."""
    CALLS.append({"presented": api_key, "action": action, "resource": resource})
    return {"ok": True, "caller": api_key}

for name in sorted(AGENTS):
    downstream(AGENTS[name]["key"], "read", "reports")
downstream(SHARED_KEY, "delete", "prod.customers")     # one of them did this

print("what the downstream service recorded:")
for c in CALLS:
    print(f"   caller={c['presented']}  {c['action']:7s} {c['resource']}")

incident = [c for c in CALLS if c["action"] == "delete"]
candidates = sorted(AGENTS)
print(f"\nincident: {incident[0]['action']} on {incident[0]['resource']}")
print(f"which agent did it? candidates: {candidates}")
print(f"distinguishable from the record? {len({c['presented'] for c in CALLS}) > 1}")

print("\ncontainment options:")
print(f"   rotate {SHARED_KEY} -> stops the incident, and stops "
      f"{len(AGENTS)} agents including {len(AGENTS)-1} innocent ones")
print("   rotate only the culprit -> not available; there is no 'only'")
print()
print("No attacker forged anything. Impersonation is the resting state of a")
print("system where identity was never per-workload.")
assert len({c["presented"] for c in CALLS}) == 1

## What you just proved

Three agents share one credential, so the downstream record shows a single caller on every line. When one deletes a production table the culprit is not recoverable from the record, and the only containment available stops all three.

## Your turn

Count the distinct credentials across your agents and divide by the number of agents. Any answer below one is this risk, and the number tells you how many innocent agents a revocation takes down.

---

**Next → [A1.8 · Unexpected code execution](https://spbreed.github.io/cyber-commons/lessons/A1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*